# B4 — A neural net in 100 lines of numpy

**From:** B2's straight-line classifier  **To:** a trained neural network you built, verified, and can no longer be mystified by — plus the bridge to the PyTorch you ran at Fujitsu.

B2's model draws a *line*. Some truths are not line-shaped. Today's data: two interleaved arcs (the classic "moons") — no line can separate them. The fix: stack **linear → nonlinearity → linear**, and the machine can bend.

New words, defined: a **layer** is a matrix multiply (+ bias); **hidden units** are the intermediate values between layers; an **activation** is the nonlinear squish between layers — ours is **ReLU**: max(0, x), brutally simple. Without the nonlinearity, stacked layers collapse algebraically into one line again (multiply two matrices, get a matrix) — the nonlinearity is *load-bearing*.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("ready · repo:", ROOT.name)

n = 240
t1 = rng.uniform(0, np.pi, n // 2); t2 = rng.uniform(0, np.pi, n // 2)
arc1 = np.column_stack([np.cos(t1), np.sin(t1)]) + rng.normal(0, 0.12, (n // 2, 2))
arc2 = np.column_stack([1 - np.cos(t2), 0.4 - np.sin(t2)]) + rng.normal(0, 0.12, (n // 2, 2))
X = np.vstack([arc1, arc2]); y = np.array([0] * (n // 2) + [1] * (n // 2))
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.7, label="class 0")
ax.scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.7, marker="x", label="class 1")
ax.set_title("two moons - no straight line separates these"); ax.legend(); plt.show()

## The forward pass = shapes flowing through matrices
Architecture: input (n,2) → layer 1 weights (2,16) → ReLU → layer 2 weights (16,1) → sigmoid → probability. Follow the shapes; that is 80% of understanding any network.

In [ ]:
sig = lambda z: 1 / (1 + np.exp(-z))
H = 16
W1 = rng.normal(0, 0.8, (2, H)); b1 = np.zeros(H)
W2 = rng.normal(0, 0.8, (H, 1)); b2 = np.zeros(1)

def forward(X):
    z1 = X @ W1 + b1          # (n,2)@(2,16) -> (n,16)
    a1 = np.maximum(0, z1)    # ReLU: negatives become 0
    z2 = a1 @ W2 + b2         # (n,16)@(16,1) -> (n,1)
    return z1, a1, z2, sig(z2).ravel()

_, _, _, p = forward(X)
print("untrained predictions hover near chance:", p[:6].round(2))

## Backward: the chain rule, industrialized
B1's chain rule, applied layer by layer from the loss backwards — each layer asks "how should MY weights nudge, given how my output should nudge?" The algebra is mechanical (calculus homework, skippable today per our agreement); the **code + a numerical check** is the trust we need:

In [ ]:
def backward(X, y, z1, a1, p):
    n = len(y)
    dz2 = (p - y).reshape(-1, 1) / n            # cross-entropy + sigmoid: famously clean gradient
    dW2 = a1.T @ dz2; db2 = dz2.sum(0)
    dz1 = (dz2 @ W2.T) * (z1 > 0)               # ReLU gate: gradient flows only where it was active
    dW1 = X.T @ dz1; db1 = dz1.sum(0)
    return dW1, db1, dW2, db2

def loss_fn(p, y):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

z1, a1, z2, p = forward(X)
dW1, db1, dW2, db2 = backward(X, y, z1, a1, p)
i, j, eps = 0, 3, 1e-5
W1[i, j] += eps; _, _, _, p_hi = forward(X); L_hi = loss_fn(p_hi, y)
W1[i, j] -= 2 * eps; _, _, _, p_lo = forward(X); L_lo = loss_fn(p_lo, y)
W1[i, j] += eps
print(f"gradient via backward: {dW1[i, j]:.6f}   via nudge-test: {(L_hi - L_lo) / (2 * eps):.6f}")
print("they match -> the backward code is telling the truth")

That check is the same finite-difference honesty from B1, now policing a real network. Professionals do exactly this when implementing layers by hand.

## Train, and watch the boundary bend

In [ ]:
lr = 0.6
losses = []
for step in range(1500):
    z1, a1, z2, p = forward(X)
    losses.append(loss_fn(p, y))
    dW1, db1, dW2, db2 = backward(X, y, z1, a1, p)
    W1 -= lr * dW1; b1 -= lr * db1; W2 -= lr * dW2; b2 -= lr * db2
print(f"final train accuracy: {((p > .5) == y).mean():.0%}")

gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, 250), np.linspace(-1.3, 1.7, 250))
_, _, _, pp = forward(np.column_stack([gx.ravel(), gy.ravel()]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses); axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].set_title("training")
axes[1].contourf(gx, gy, pp.reshape(gx.shape), levels=20, cmap="RdBu_r", alpha=0.7)
axes[1].scatter(X[y == 0, 0], X[y == 0, 1], s=12)
axes[1].scatter(X[y == 1, 0], X[y == 1, 1], s=12, marker="x")
axes[1].set_title("the boundary BENDS - hello nonlinearity"); plt.tight_layout(); plt.show()

A curved decision field from nothing but matrices, max(0,·), and B1's loop. **PREDICT, then run:** with 2 hidden units instead of 16 — what does the boundary look like? With 64 — better or just *busier*?

In [ ]:
def train_width(H, steps=1500, lr=0.6):
    W1 = rng.normal(0, 0.8, (2, H)); b1 = np.zeros(H)
    W2 = rng.normal(0, 0.8, (H, 1)); b2 = np.zeros(1)
    for _ in range(steps):
        z1 = X @ W1 + b1; a1 = np.maximum(0, z1); p = sig((a1 @ W2 + b2)).ravel()
        dz2 = (p - y).reshape(-1, 1) / len(y)
        dW2 = a1.T @ dz2; db2 = dz2.sum(0)
        dz1 = (dz2 @ W2.T) * (z1 > 0)
        W1 -= lr * (X.T @ dz1); b1 -= lr * dz1.sum(0); W2 -= lr * dW2; b2 -= lr * db2
    def f(Q):
        return sig((np.maximum(0, Q @ W1 + b1) @ W2 + b2)).ravel()
    return f

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, H in zip(axes, [2, 16, 64]):
    f = train_width(H)
    ax.contourf(gx, gy, f(np.column_stack([gx.ravel(), gy.ravel()])).reshape(gx.shape),
                levels=20, cmap="RdBu_r", alpha=0.7)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=8); ax.scatter(X[y == 1, 0], X[y == 1, 1], s=8, marker="x")
    ax.set_title(f"{H} hidden units")
plt.tight_layout(); plt.show()

Two units underfit (not enough bend); sixteen is graceful; sixty-four traces noise wiggles — B2's overfitting lesson, now in curve form. Capacity is a dial, not a virtue.

## The bridge to what you already ran
At Fujitsu you wrote something like: `loss.backward(); optimizer.step()`. Decoded against today:
- `model(x)` — our `forward` (matrices + activations)
- `loss.backward()` — our `backward` cell, except **autograd** derives it mechanically for any architecture (that is PyTorch's entire magic trick)
- `optimizer.step()` — our `W -= lr * dW` lines (fancier optimizers like Adam add per-weight adaptive learning rates)
- GPU — these same matrix multiplies, thousands at once

And an LLM? *This machine*, with ~10⁹–10¹² parameters, a smarter wiring diagram (attention — Part C), trained by this exact loop on next-token cross-entropy (B2). You have now personally built every conceptual component except attention. The mystery budget is almost spent.

## Self-check
1. Why is the nonlinearity non-negotiable? (The algebraic reason.)
2. What does ReLU do during backward, not just forward?
3. What does the numerical gradient check prove, and what does it not prove?
4. Width 64 hit 100% train accuracy — your reaction, verbatim from B2?
5. **Gotcha:** "PyTorch trains models." What does PyTorch actually automate, in one sentence?

<details><summary>Answers</summary>

1. Stacked linear maps compose into a single linear map (matrix product) — without a nonlinearity, depth buys nothing and the boundary stays a line.
2. It gates gradients: where the unit was inactive (z≤0), zero gradient flows back — only active paths learn.
3. That backward computes the true derivative of THIS loss for THIS code; not that the model is good, the data sensible, or training will converge.
4. "And on data it has never seen?" — train-perfect means little; check the generalization gap.
5. It automates the backward pass (autograd) and bookkeeping around the same loop you wrote today — the learning is still loss + gradient + step.
</details>